[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bethavila17/dncp-riesgo-contrataciones/blob/main/notebooks/04_redes_y_tendencias.ipynb)

# 4. Análisis de redes y pronóstico de tendencias
**Proyecto III Big Data — Maestría en Ciencia de Datos (UCOM)** · Bethania Avila, Arturo Fernández, Gustavo Bogado

**Redes:** dependencia entidad–proveedor (HHI) y red de co-participación de oferentes (comunidades de Louvain, patrones de rotación y de oferta de cobertura; Huber e Imhof, 2019). Son señales para revisión documental, no prueba de colusión.

**Tendencias (objetivo 4 del TP: prever tendencias):** se comparan Holt-Winters, SARIMA y un modelo ingenuo estacional con *backtesting*, y el mejor pronostica los próximos 6 meses con intervalos de predicción.

In [ ]:
# Configuración del entorno (Google Colab o local)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/bethavila17/dncp-riesgo-contrataciones.git"  # <- URL del repositorio en GitHub
USAR_DRIVE = False  # True: guarda el data lake en Google Drive y no repite descargas entre sesiones

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/dncp-riesgo-contrataciones")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements-colab.txt")], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["DNCP_DATA_DIR"] = "/content/drive/MyDrive/dncp_data_lake"
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dncp_riesgo import graficos as gr
from dncp_riesgo.pipeline import asegurar_datos, preparar

cfg, lake = preparar()
gr.estilo()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| años:", cfg["fuentes"]["ocp_registry"]["anios"])

In [ ]:
asegurar_datos(lake, cfg, hasta="redes")
from dncp_riesgo.pipeline import etapa_redes, etapa_series
red = etapa_redes(lake, cfg)
pares = red["pares"]
print(f"{red['grafo'].number_of_nodes():,} proveedores | {len(pares):,} pares con ≥5 co-participaciones | "
      f"rotación: {pares.rotacion.sum()} | cobertura: {pares.cobertura.sum()} | comunidades: {len(red['comunidades'])}".replace(",", "."))

## 4.1 Cribado de pares de proveedores (llamados con un único adjudicado)

In [ ]:
p = pares[pares.juntos_ganador_unico >= 10].copy()
p["gana_max"] = p[["gana_a", "gana_b"]].max(axis=1) / p.juntos_ganador_unico * 100
p["gana_min"] = p[["gana_a", "gana_b"]].min(axis=1) / p.juntos_ganador_unico * 100
fig, ax = plt.subplots(figsize=(5.5, 4))
normal = p[p.puntaje_colusion == 0]; marcados = p[p.puntaje_colusion > 0]
ax.scatter(normal.gana_max, normal.gana_min, s=8 + normal.juntos_ganador_unico * 0.5, facecolors="none", edgecolors=gr.TENUE, lw=0.6, label="resto de los pares")
ax.scatter(marcados.gana_max, marcados.gana_min, s=8 + marcados.juntos_ganador_unico * 0.5, color=gr.NIVELES["Alto"], label="rotación o cobertura")
ax.set_xlabel("victorias de la empresa que más gana (%)"); ax.set_ylabel("victorias de la otra (%)"); ax.legend(loc="upper left")
ax.set_title(f"Cribado de {len(p)} pares con ≥10 llamados compartidos")
gr.guardar(fig, "fig10_cribado_pares"); plt.show()
pares[pares.puntaje_colusion > 0].head(10)[["nombre_a", "nombre_b", "juntos_ganador_unico", "gana_a", "gana_b", "gana_otro", "rotacion", "cobertura"]]

## 4.2 Comunidades de co-participación

In [ ]:
from dncp_riesgo.modelos.redes import dibujar_comunidad
display(red["comunidades"].head(8))
fig, ax = plt.subplots(figsize=(9, 6))
dibujar_comunidad(red["grafo"], red["metricas"], int(red["comunidades"].iloc[0]["comunidad"]), pares, ax=ax)
gr.guardar(fig, "fig11_comunidad"); plt.show()

## 4.3 Concentración de las entidades (HHI)

In [ ]:
hhi = red["hhi"]
display(hhi.groupby("concentracion").agg(entidades=("entidad", "count"), monto_mm=("monto_mm", "sum")).round(0))
hhi[hhi.monto_mm > 10].head(10)

## 4.4 Pronóstico de tendencias

In [ ]:
from dncp_riesgo.modelos.series import graficar_pronostico, tabla_backtest
pron = etapa_series(lake, cfg)
tabla_backtest(pron)

In [ ]:
titulos = {"n_procesos": "Convocatorias por mes", "monto_adjudicado_mm": "Monto adjudicado (miles de millones Gs constantes)",
           "pct_oferente_unico": "Proporción de procesos con oferente único"}
fig, ejes = plt.subplots(3, 1, figsize=(8, 9))
for ax, (col, titulo) in zip(ejes, titulos.items()):
    graficar_pronostico(pron[col], titulo, ax=ax)
fig.tight_layout(); gr.guardar(fig, "fig12_pronosticos"); plt.show()

In [ ]:
pron["pct_oferente_unico"]["pronostico"].round(3)